# 11E — Robustness and fair seed-design comparison

Notebook 11D found that directional critical-minerals scores—not unsigned thematic proximity—were associated with forward returns in the two complete 2024 windows. This notebook stress-tests that result without changing the frozen text labels or embedding scores.

The tests address seed-count differences, industry and firm-characteristic confounding, portfolio cutoffs, direct-text versus embedding-expanded portfolios, permutation inference, and false-discovery-rate adjustment. Results remain exploratory because only two complete formation periods are locally available.

## 1. Verify the frozen 11D handoff

Every upstream artifact is hash-checked before this notebook constructs robustness tests.

In [1]:
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
from IPython.display import display

HERE=Path.cwd().resolve()
if HERE.name=='quarterly_sec_pilot': PILOT_ROOT,GEO_ROOT,ROOT=HERE,HERE.parent,HERE.parents[1]
elif HERE.name=='geopolitical_exposure': GEO_ROOT,ROOT=HERE,HERE.parent;PILOT_ROOT=GEO_ROOT/'quarterly_sec_pilot'
elif HERE.name=='Asset_Embeddings': ROOT=HERE;GEO_ROOT=ROOT/'geopolitical_exposure';PILOT_ROOT=GEO_ROOT/'quarterly_sec_pilot'
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve();GEO_ROOT=ROOT/'geopolitical_exposure';PILOT_ROOT=GEO_ROOT/'quarterly_sec_pilot'
UP=PILOT_ROOT/'quarterly_portfolio_evaluation'; UPR=UP/'results'; UPM=UP/'manifests'
SEED_ROOT=PILOT_ROOT/'three_seed_specifications';OUT=PILOT_ROOT/'robustness_seed_design';RESULTS=OUT/'results';MANIFESTS=OUT/'manifests'
for p in [RESULTS,MANIFESTS]:p.mkdir(parents=True,exist_ok=True)
def sha256(p):return hashlib.sha256(p.read_bytes()).hexdigest()
up_manifest=json.loads((UPM/'evaluation_manifest.json').read_text());up_inventory=pd.read_parquet(UPM/'output_inventory.parquet')
for row in up_inventory.itertuples():
 p=UPR/row.artifact;assert p.exists() and sha256(p)==row.sha256
scores=pd.read_parquet(UPR/'quarterly_decomposition_scores.parquet')
forward=pd.read_parquet(UPR/'publication_lagged_forward_returns.parquet')
seeds=pd.read_parquet(SEED_ROOT/'processed'/'automated_quarterly_seed_panel.parquet')
print('Verified 11D:',up_manifest['sha256'])

Verified 11D: fec4b1cd198968635d3912431cfc3df9b5c9fa109435788f45e39574769c511d


## 2. Freeze the primary robustness sample and controls

The primary outcome sample is limited to Q1 and Q2 complete three-month forward windows. Controls are measured at the corresponding embedding vintage. SIC2 uses the already-downloaded 2024Q4 CRSP names snapshot and is treated as a descriptive industry control.

In [2]:
PRIMARY_VINTAGES=['2024q1','2024q2'];MODELS=['rs_binary','rs_l_min'];SPECS=['broad','sector_enriched','combined']
dynamic=scores[(scores.decomposition=='dynamic')&(~scores.is_any_seed)].copy()
primary=dynamic.merge(forward.query('complete_window')[['vintage','permco','forward_return']],on=['vintage','permco'],how='inner',validate='many_to_one')
controls=[]
for vintage in PRIMARY_VINTAGES:
 c=pd.read_parquet(ROOT/'remaining_benchmarks'/'return_comovement'/'characteristics'/f'{vintage}.parquet').copy();c['vintage']=vintage;controls.append(c)
controls=pd.concat(controls,ignore_index=True)
names=pd.read_parquet(ROOT/'data'/'crosswalks'/'crsp'/'year=2024'/'quarter=4'/'stocknames_2024q4.parquet')
names['siccd']=pd.to_numeric(names.siccd,errors='coerce');industry=(names.dropna(subset=['permco']).sort_values(['permco','nameenddt']).drop_duplicates('permco',keep='last')[['permco','siccd']])
industry['permco']=industry.permco.astype(int);industry['sic2']=(industry.siccd//100).astype('Int64')
primary=primary.merge(controls,on=['vintage','permco'],how='left',validate='many_to_one').merge(industry[['permco','sic2']],on='permco',how='left',validate='many_to_one')
control_columns=['beta_60m_paper','log_market_equity','log_book_to_market','at_gr1_paper','gp_at_paper','momentum_12_1']
for c in control_columns:primary[c]=pd.to_numeric(primary[c],errors='coerce');primary[c]=primary.groupby('vintage')[c].transform(lambda x:x.fillna(x.median()))
primary['size_quintile']=primary.groupby('vintage').log_market_equity.transform(lambda x:pd.qcut(x.rank(method='first'),5,labels=False,duplicates='drop'))
primary['permutation_stratum']=primary.sic2.astype('string').fillna('missing')+'|'+primary.size_quintile.astype('Int64').astype('string')
sample_audit=(primary.groupby(['vintage','embedding_model','seed_specification','target'],as_index=False).agg(firms=('permco','nunique'),return_mean=('forward_return','mean'),control_complete=('log_market_equity','count')))
sample_audit.to_parquet(MANIFESTS/'robustness_sample_audit.parquet',index=False);display(sample_audit)

,vintage,embedding_model,seed_specification,target,firms,return_mean,control_complete
0,2024q1,rs_binary,broad,relatedness_score,1242,0.067366,1242
1,2024q1,rs_binary,broad,signed_score,1242,0.067366,1242
2,2024q1,rs_binary,combined,relatedness_score,1242,0.067366,1242
3,2024q1,rs_binary,combined,signed_score,1242,0.067366,1242
4,2024q1,rs_binary,sector_enriched,relatedness_score,1242,0.067366,1242
5,2024q1,rs_binary,sector_enriched,signed_score,1242,0.067366,1242
6,2024q1,rs_l_min,broad,relatedness_score,1242,0.067366,1242
7,2024q1,rs_l_min,broad,signed_score,1242,0.067366,1242
8,2024q1,rs_l_min,combined,relatedness_score,1242,0.067366,1242
9,2024q1,rs_l_min,combined,signed_score,1242,0.067366,1242


## 3. Characteristic- and industry-adjusted associations

Frisch–Waugh–Lovell residualization removes the six firm characteristics and SIC2 fixed effects from both returns and the propagated score. The reported coefficient is standardized. Inference uses 500 return permutations within SIC2 × size-quintile strata.

In [3]:
def residualize(values,frame):
 x=frame[control_columns].to_numpy(float);x=(x-x.mean(0))/np.where(x.std(0)>0,x.std(0),1)
 d=pd.get_dummies(frame.sic2.astype('string').fillna('missing'),drop_first=True,dtype=float).to_numpy()
 X=np.column_stack([np.ones(len(frame)),x,d]);y=np.asarray(values,float)
 with np.errstate(over='ignore',invalid='ignore',divide='ignore'):
  fitted=X@np.linalg.lstsq(X,y,rcond=None)[0]
 residual=y-fitted
 if not np.isfinite(residual).all():raise FloatingPointError('Non-finite control residual')
 return residual
def stratified_shuffle(values,strata,rng):
 out=np.asarray(values).copy()
 for label in pd.unique(strata):
  idx=np.flatnonzero(np.asarray(strata)==label);out[idx]=rng.permutation(out[idx])
 return out
rng=np.random.default_rng(20240905);adjusted=[];REPS=500
for keys,g in primary.groupby(['vintage','embedding_model','seed_specification','target']):
 g=g.reset_index(drop=True);sr=residualize(g.propagated_score,g);yr=residualize(g.forward_return,g)
 sr=sr/np.std(sr);denominator=float(np.dot(sr,sr));beta=float(np.dot(sr,yr)/denominator);rho=pd.Series(sr).corr(pd.Series(yr),method='spearman');null=[]
 for rep in range(REPS):
  shuffled=stratified_shuffle(g.forward_return,g.permutation_stratum,rng);null.append(float(np.dot(sr,shuffled)/denominator))
 adjusted.append(dict(zip(['vintage','embedding_model','seed_specification','target'],keys),firms=len(g),
  standardized_beta=beta,partial_spearman=rho,permutation_p_two_sided=(1+np.sum(np.abs(null)>=abs(beta)))/(REPS+1)))
adjusted=pd.DataFrame(adjusted);adjusted.to_parquet(RESULTS/'characteristic_industry_adjusted_results.parquet',index=False)
display(adjusted.sort_values(['target','embedding_model','seed_specification','vintage']))

,vintage,embedding_model,seed_specification,target,firms,standardized_beta,partial_spearman,permutation_p_two_sided
0,2024q1,rs_binary,broad,relatedness_score,1242,-0.007102,-0.027144,0.291417
12,2024q2,rs_binary,broad,relatedness_score,1242,-0.002917,0.004934,0.638723
2,2024q1,rs_binary,combined,relatedness_score,1242,-0.003875,-0.001090,0.469062
14,2024q2,rs_binary,combined,relatedness_score,1242,-0.002668,0.006854,0.590818
4,2024q1,rs_binary,sector_enriched,relatedness_score,1242,-0.001927,0.016062,0.606786
16,2024q2,rs_binary,sector_enriched,relatedness_score,1242,-0.002274,0.004983,0.646707
6,2024q1,rs_l_min,broad,relatedness_score,1242,-0.004530,-0.033324,0.339321
18,2024q2,rs_l_min,broad,relatedness_score,1242,-0.002109,-0.002765,0.670659
8,2024q1,rs_l_min,combined,relatedness_score,1242,-0.000528,0.013414,0.900200
20,2024q2,rs_l_min,combined,relatedness_score,1242,0.000351,0.004759,0.930140


## 4. Portfolio-cutoff sensitivity

The same frozen ranking is evaluated using symmetric 5%, 10%, and 20% tails. A credible directional result should not depend entirely on one arbitrary breakpoint.

In [4]:
cutoff_rows=[]
for keys,g in primary.groupby(['vintage','embedding_model','seed_specification','target']):
 rank=g.propagated_score.rank(pct=True,method='average')
 for tail in [.05,.10,.20]:
  hi=g.loc[rank>=1-tail,'forward_return'];lo=g.loc[rank<=tail,'forward_return']
  cutoff_rows.append(dict(zip(['vintage','embedding_model','seed_specification','target'],keys),tail_fraction=tail,
   high_firms=len(hi),low_firms=len(lo),high_return=hi.mean(),low_return=lo.mean(),high_minus_low=hi.mean()-lo.mean()))
cutoffs=pd.DataFrame(cutoff_rows);cutoffs.to_parquet(RESULTS/'portfolio_cutoff_sensitivity.parquet',index=False)
display(cutoffs.groupby(['target','embedding_model','seed_specification','tail_fraction'],as_index=False).agg(
 quarters=('vintage','nunique'),mean_spread=('high_minus_low','mean')))

,target,embedding_model,seed_specification,tail_fraction,quarters,mean_spread
0,relatedness_score,rs_binary,broad,0.05,2,0.000274
1,relatedness_score,rs_binary,broad,0.10,2,-0.010149
2,relatedness_score,rs_binary,broad,0.20,2,-0.024606
3,relatedness_score,rs_binary,combined,0.05,2,-0.037218
4,relatedness_score,rs_binary,combined,0.10,2,-0.033880
5,relatedness_score,rs_binary,combined,0.20,2,-0.022246
6,relatedness_score,rs_binary,sector_enriched,0.05,2,-0.083200
7,relatedness_score,rs_binary,sector_enriched,0.10,2,-0.053089
8,relatedness_score,rs_binary,sector_enriched,0.20,2,-0.021156
9,relatedness_score,rs_l_min,broad,0.05,2,-0.003602


## 5. Direct-text portfolio benchmark

This benchmark invests only in text-labelled seed firms. It is not an incremental-discovery portfolio and has a much smaller universe, so it is reported separately rather than treated as a head-to-head test with the embedding expansion.

In [5]:
direct=[]
for spec in SPECS:
 for vintage in PRIMARY_VINTAGES:
  s=seeds[(seeds.seed_specification==spec)&(seeds.vintage==vintage)&(seeds.channel=='critical_minerals_overall')].drop_duplicates('firm_id')
  f=forward[(forward.vintage==vintage)&forward.complete_window][['permco','forward_return']]
  z=s.merge(f,left_on='firm_id',right_on='permco',how='inner');pos=z[z.signed_score>0];neg=z[z.signed_score<0]
  direct.append({'vintage':vintage,'seed_specification':spec,'available_seed_firms':len(z),'positive_firms':len(pos),'negative_firms':len(neg),
   'positive_return':pos.forward_return.mean(),'negative_return':neg.forward_return.mean(),
   'positive_minus_negative':pos.forward_return.mean()-neg.forward_return.mean()})
direct=pd.DataFrame(direct);direct.to_parquet(RESULTS/'direct_text_seed_portfolios.parquet',index=False);display(direct)

,vintage,seed_specification,available_seed_firms,positive_firms,negative_firms,positive_return,negative_return,positive_minus_negative
0,2024q1,broad,5,1,4,-0.132554,-0.054650,-0.077904
1,2024q2,broad,6,2,4,0.100640,0.159070,-0.058430
2,2024q1,sector_enriched,16,2,11,-0.103785,-0.057202,-0.046583
3,2024q2,sector_enriched,16,2,11,0.261471,0.193709,0.067762
4,2024q1,combined,21,3,15,-0.113375,-0.056521,-0.056853
5,2024q2,combined,22,4,15,0.181055,0.184472,-0.003416


## 6. Equal-count enriched-seed experiment

The enriched arm normally has more seeds than the broad arm. For each quarter, model, and target, this test repeatedly samples the same number of nonzero enriched seeds as are available in the broad arm, then rebuilds the embedding score. It isolates seed volume, while deliberately leaving direction composition as a feature of each sampling design. Exact sign matching is infeasible in Q1 because no enriched positive-direction seed survives that quarter's embedding universe.

In [6]:
def embedding(vintage,model):
 p=ROOT/'active_passive_extension'/'benchmarks'/'all_s12_control'/vintage/model/'asset_embeddings_10d.parquet';f=pd.read_parquet(p).sort_values('permco').reset_index(drop=True);f['permco']=f.permco.astype(int)
 dims=[c for c in f if c.startswith('embedding_')];x=f[dims].to_numpy(float);n=np.linalg.norm(x,axis=1,keepdims=True);return f,np.divide(x,n,out=np.zeros_like(x),where=n>0)
def score_from_seed_rows(f,x,s,target):
 idx=pd.Series(f.index,index=f.permco);s=s[s.firm_id.isin(idx.index)].drop_duplicates('firm_id');w=s[target].to_numpy(float);den=np.abs(w).sum()
 with np.errstate(over='ignore',invalid='ignore',divide='ignore'):out=np.zeros(len(f)) if den==0 else (x@x[idx.loc[s.firm_id].to_numpy()].T)@w/den
 if not np.isfinite(out).all():raise FloatingPointError('Non-finite score')
 return out
rng=np.random.default_rng(20240906);MATCHED_REPS=500;matched=[]
for vintage in PRIMARY_VINTAGES:
 outcome=forward[(forward.vintage==vintage)&forward.complete_window].set_index('permco').forward_return
 broad=seeds[(seeds.seed_specification=='broad')&(seeds.vintage==vintage)&(seeds.channel=='critical_minerals_overall')]
 enriched=seeds[(seeds.seed_specification=='sector_enriched')&(seeds.vintage==vintage)&(seeds.channel=='critical_minerals_overall')]
 for model in MODELS:
  f,x=embedding(vintage,model);eligible=f.permco.isin(outcome.index)&~f.permco.isin(set(seeds[seeds.seed_specification=='combined'].firm_id));eval_idx=np.flatnonzero(eligible);ret=outcome.loc[f.loc[eval_idx,'permco']].to_numpy()
  for target in ['relatedness_score','signed_score']:
   b=broad[broad.firm_id.isin(f.permco)&broad[target].ne(0)];e=enriched[enriched.firm_id.isin(f.permco)&enriched[target].ne(0)]
   for rep in range(MATCHED_REPS):
    sampled=e.sample(n=len(b),replace=False,random_state=int(rng.integers(2**31-1)))
    sc=score_from_seed_rows(f,x,sampled,target)[eval_idx];pct=pd.Series(sc).rank(pct=True).to_numpy();spread=ret[pct>=.9].mean()-ret[pct<=.1].mean()
    matched.append({'vintage':vintage,'embedding_model':model,'target':target,'repetition':rep,'sampled_seeds':len(sampled),
                    'rank_correlation':pd.Series(sc).corr(pd.Series(ret),method='spearman'),'top_minus_bottom':spread})
matched=pd.DataFrame(matched);matched.to_parquet(RESULTS/'equal_count_enriched_seed_results.parquet',index=False)
matched_summary=matched.groupby(['vintage','embedding_model','target'],as_index=False).agg(
 sampled_seeds=('sampled_seeds','median'),median_rank_correlation=('rank_correlation','median'),
 p10_rank_correlation=('rank_correlation',lambda x:x.quantile(.1)),p90_rank_correlation=('rank_correlation',lambda x:x.quantile(.9)),
 median_spread=('top_minus_bottom','median'),p10_spread=('top_minus_bottom',lambda x:x.quantile(.1)),p90_spread=('top_minus_bottom',lambda x:x.quantile(.9)))
matched_summary.to_parquet(RESULTS/'equal_count_enriched_seed_summary.parquet',index=False);display(matched_summary)

,vintage,embedding_model,target,sampled_seeds,median_rank_correlation,p10_rank_correlation,p90_rank_correlation,median_spread,p10_spread,p90_spread
0,2024q1,rs_binary,relatedness_score,5.0,-0.017359,-0.074284,0.022812,-0.018897,-0.060377,0.013109
1,2024q1,rs_binary,signed_score,5.0,0.005223,-0.022984,0.063005,0.013097,-0.015350,0.051187
2,2024q1,rs_l_min,relatedness_score,5.0,-0.016550,-0.055241,0.032201,-0.004410,-0.035408,0.027322
3,2024q1,rs_l_min,signed_score,5.0,0.007306,-0.038636,0.050311,-0.001831,-0.033228,0.032584
4,2024q2,rs_binary,relatedness_score,6.0,-0.012686,-0.036694,0.015197,-0.000835,-0.027459,0.021983
5,2024q2,rs_binary,signed_score,6.0,0.009881,-0.018446,0.034986,0.009730,-0.016511,0.034275
6,2024q2,rs_l_min,relatedness_score,6.0,-0.031526,-0.059421,0.003048,-0.012261,-0.074526,0.029698
7,2024q2,rs_l_min,signed_score,6.0,0.024146,-0.006948,0.055782,0.023070,-0.017909,0.080869


## 7. False-discovery-rate correction

Benjamini–Hochberg q-values are calculated across the frozen dynamic model comparisons. This does not solve the two-quarter limitation, but it prevents isolated nominal p-values from being overstated.

In [7]:
permutation=pd.read_parquet(UPR/'matched_permutation_results.parquet').query("decomposition=='dynamic'").copy()
def bh(p):
 p=np.asarray(p,float);order=np.argsort(p);ranked=p[order];q=np.minimum.accumulate((ranked*len(p)/np.arange(1,len(p)+1))[::-1])[::-1];out=np.empty(len(p));out[order]=np.minimum(q,1);return out
permutation['rank_q_bh']=bh(permutation.rank_p_two_sided);permutation['spread_q_bh']=bh(permutation.spread_p_two_sided)
permutation.to_parquet(RESULTS/'dynamic_results_multiple_testing_adjusted.parquet',index=False)
display(permutation.sort_values(['target','spread_q_bh','rank_q_bh']))

,vintage,embedding_model,seed_specification,target,decomposition,actual_rank_correlation,rank_p_two_sided,actual_top_minus_bottom,spread_p_two_sided,rank_q_bh,spread_q_bh
6,2024q1,rs_binary,combined,relatedness_score,dynamic,-0.089193,0.001996,-0.055091,0.013972,0.009581,0.047904
12,2024q1,rs_binary,sector_enriched,relatedness_score,dynamic,-0.058182,0.051896,-0.084742,0.001996,0.103792,0.047904
24,2024q1,rs_l_min,combined,relatedness_score,dynamic,-0.043323,0.139721,-0.056045,0.009980,0.223553,0.047904
30,2024q1,rs_l_min,sector_enriched,relatedness_score,dynamic,-0.022686,0.423154,-0.060531,0.011976,0.461622,0.047904
0,2024q1,rs_binary,broad,relatedness_score,dynamic,-0.087910,0.001996,-0.026883,0.293413,0.009581,0.469461
66,2024q2,rs_l_min,sector_enriched,relatedness_score,dynamic,-0.047113,0.083832,-0.027150,0.325349,0.154767,0.488024
48,2024q2,rs_binary,sector_enriched,relatedness_score,dynamic,-0.032735,0.233533,-0.021436,0.415170,0.297510,0.586122
18,2024q1,rs_l_min,broad,relatedness_score,dynamic,-0.087989,0.001996,-0.016334,0.457086,0.009581,0.609448
42,2024q2,rs_binary,combined,relatedness_score,dynamic,-0.035069,0.235529,-0.012669,0.670659,0.297510,0.804790
60,2024q2,rs_l_min,combined,relatedness_score,dynamic,-0.038114,0.221557,0.006133,0.826347,0.297510,0.901470


## 8. Decision table and reproducible handoff

The decision table summarizes robustness rather than declaring success from a single specification. Extending returns through 2025 remains necessary for confirmatory inference.

In [8]:
signed_adjusted=adjusted[adjusted.target=='signed_score'];signed_cutoffs=cutoffs[cutoffs.target=='signed_score']
decisions=pd.DataFrame([
 {'test':'signed coefficient positive after characteristics and SIC2','value':float((signed_adjusted.standardized_beta>0).mean()),'interpretation':'share of specifications'},
 {'test':'signed spread positive across 5/10/20 percent tails','value':float((signed_cutoffs.high_minus_low>0).mean()),'interpretation':'share of quarter-model-seed-cutoff tests'},
 {'test':'dynamic signed tests surviving BH at 10 percent','value':int(((permutation.target=='signed_score')&((permutation.rank_q_bh<.10)|(permutation.spread_q_bh<.10))).sum()),'interpretation':'model-quarter tests'},
 {'test':'complete independent forward periods','value':len(PRIMARY_VINTAGES),'interpretation':'insufficient for confirmatory claim'},
])
decisions.to_parquet(RESULTS/'robustness_decision_table.parquet',index=False);display(decisions)
artifacts=['characteristic_industry_adjusted_results.parquet','portfolio_cutoff_sensitivity.parquet','direct_text_seed_portfolios.parquet',
 'equal_count_enriched_seed_results.parquet','equal_count_enriched_seed_summary.parquet','dynamic_results_multiple_testing_adjusted.parquet','robustness_decision_table.parquet']
inv=[]
for name in artifacts:
 p=RESULTS/name;inv.append({'artifact':name,'rows':len(pd.read_parquet(p)),'bytes':p.stat().st_size,'sha256':sha256(p)})
inv=pd.DataFrame(inv);inv.to_parquet(MANIFESTS/'output_inventory.parquet',index=False)
manifest={'stage':'quarterly_sec_embedding_robustness_v1','upstream_evaluation_sha256':up_manifest['sha256'],
 'primary_vintages':PRIMARY_VINTAGES,'controls':control_columns+['sic2'],'industry_snapshot':'2024q4',
 'portfolio_tail_fractions':[.05,.10,.20],'permutation_repetitions':REPS,'matched_seed_repetitions':MATCHED_REPS,
 'fdr_method':'Benjamini-Hochberg','confirmatory_status':False,'reason':'only two complete forward periods',
 'artifacts':inv.to_dict('records')}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest();(MANIFESTS/'robustness_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
display(inv);print('Robustness handoff:',manifest['sha256'])

,test,value,interpretation
0,signed coefficient positive after characterist...,0.916667,share of specifications
1,signed spread positive across 5/10/20 percent ...,0.944444,share of quarter-model-seed-cutoff tests
2,dynamic signed tests surviving BH at 10 percent,10.000000,model-quarter tests
3,complete independent forward periods,2.000000,insufficient for confirmatory claim


,artifact,rows,bytes,sha256
0,characteristic_industry_adjusted_results.parquet,24,6031,415aa61224f42ca82905d989b4770897437cd479b543d4...
1,portfolio_cutoff_sensitivity.parquet,72,8539,76e9b767478c500a2d13f64e382d98ae4495559f04b94b...
2,direct_text_seed_portfolios.parquet,6,5754,0a1707251afd4bb62298b2a67d778053fde8f5f18300a8...
3,equal_count_enriched_seed_results.parquet,4000,68996,2e6078a09453bfdb0114d6ae8530d9c97a78e63a5a5aa4...
4,equal_count_enriched_seed_summary.parquet,8,7023,6f769ff28253e78e06bcede4b633511605354fb1efe1c6...
5,dynamic_results_multiple_testing_adjusted.parquet,24,8283,bb2c6634ab6b0651a8c72e08cdc4d0b557f1255504de95...
6,robustness_decision_table.parquet,4,2761,7dd6b2914991d49df084d4aae731b4a92b54bdb0360212...


Robustness handoff: 5857d5a977615d560564298543b90fe67e7ca5108f29a7bd3571dcee07a69895
